# 20 — GoalAgent with Real Tools

GoalAgent that uses **web search**, **code execution**, and **URL fetching** to accomplish goals autonomously. The agent searches for real data, verifies it, and tracks success criteria.

**What you'll learn:**
1. GoalAgent.with_builtins — all tools, one line
2. GoalAgent with specific tools — web search + code exec
3. Streaming with real tool calls visible
4. Goal with memory for multi-run research

In [ ]:
from pathlib import Path
import sys, hashlib

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, AgentMemory, ConversationMemory, SemanticMemory, EntityMemory, Entity, InMemoryVectorStore, WebSearchTool, CodeExecutionTool
from shipit_agent.deep import GoalAgent, Goal
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env('bedrock')

def embed(text):
    h = hashlib.sha256(text.lower().encode()).digest()
    return [float(b) / 255.0 for b in h[:16]]

def print_event(event):
    ICONS = {"run_started": "🚀", "run_completed": "✅", "planning_started": "📋",
             "planning_completed": "📋", "step_started": "▶️", "tool_completed": "📦"}
    icon = ICONS.get(event.type, "•")
    print(f"\n{icon} {event.message}")
    output = event.payload.get("output", "")
    if output and event.type in ("tool_completed", "run_completed"):
        print(f"{'─' * 60}")
        print(output[:600])
        if len(output) > 600: print(f"... ({len(output)} chars)")
        print(f"{'─' * 60}")
    subtasks = event.payload.get("subtasks")
    if subtasks:
        for i, t in enumerate(subtasks, 1): print(f"  {i}. {t}")
    criteria = event.payload.get("criteria_met")
    if criteria is not None:
        print(f"  Criteria: {' '.join('✅' if c else '❌' for c in criteria)}")

## 1. GoalAgent.with_builtins — All Tools, One Line

The agent searches the web, reads pages, runs code — whatever it needs to meet the success criteria.

In [ ]:
# GoalAgent with ALL built-in tools — searches web, runs code, fetches pages
agent = GoalAgent.with_builtins(
    llm=llm,
    goal=Goal(
        objective="Find the latest Python version and its key new features",
        success_criteria=[
            "States the current Python version number",
            "Lists at least 3 new features",
            "Includes release date",
        ],
        max_steps=4,
    ),
)

print("=== GoalAgent.with_builtins — Real Web Research ===\n")
for event in agent.stream():
    print_event(event)

## 2. GoalAgent with Specific Tools

Give the agent only web search + code execution. No browser, no file tools — just focused capabilities.

In [ ]:
# GoalAgent with specific tools only
agent = GoalAgent(
    llm=llm,
    tools=[WebSearchTool(), CodeExecutionTool()],  # only these two
    goal=Goal(
        objective="Calculate the compound interest on $10,000 at 5% for 10 years",
        success_criteria=[
            "Shows the formula used",
            "Calculates the final amount",
            "Verifies with code execution",
        ],
        max_steps=3,
    ),
)

print("=== GoalAgent with WebSearch + CodeExecution ===\n")
for event in agent.stream():
    print_event(event)

## 3. GoalAgent with Memory — Multi-Run Research

First run researches a topic. Second run builds on it using memory.

In [ ]:
# Shared memory across runs
memory = AgentMemory(
    conversation=ConversationMemory(strategy="buffer"),
    knowledge=SemanticMemory(vector_store=InMemoryVectorStore(), embedding_fn=embed),
    entities=EntityMemory(),
)

# Run 1: Research with web search
print("=== Run 1: Research ===\n")
agent1 = GoalAgent.with_builtins(
    llm=llm,
    memory=memory,
    goal=Goal(
        objective="Find the top 3 AI agent frameworks on GitHub",
        success_criteria=["Lists 3 frameworks with star counts"],
        max_steps=3,
    ),
)
result1 = agent1.run()
print(f"Status: {result1.goal_status}")
print(f"Output: {result1.output[:300]}...")

# Save to memory
memory.add_fact(f"Research findings: {result1.output[:500]}")

# Run 2: Build on previous research
print("\n=== Run 2: Compare (uses memory from Run 1) ===\n")
agent2 = GoalAgent(
    llm=llm,
    memory=memory,  # same memory!
    goal=Goal(
        objective="Compare the frameworks you found and recommend one for production use",
        success_criteria=["References the frameworks from earlier", "Gives a recommendation"],
        max_steps=2,
    ),
)
result2 = agent2.run()
print(f"Status: {result2.goal_status}")
display(Markdown(result2.output[:500]))